# Service Kernel Gateway — Z3 Sudoku

Premier service du **pli 0** de l'Epic #19571 : exposer le carnet phare
[Z3-02-Sudoku-Python.ipynb](../Z3-02-Sudoku-Python.ipynb) en microservice REST via
Jupyter **Kernel Gateway** (mode `notebook-http`).

**Contrat** : `POST /sudoku/solve` — corps `{"puzzle": "<81 caracteres, 0 ou . pour vide>"}`
-> `{"status": "sat"|"unsat"|"invalid", "solution": "<81 caracteres>|null", "time_ms": <float>}`.

Ce carnet s'execute aussi **hors gateway** (mode local) : il sert alors de demonstration
sur le puzzle du carnet phare. Voir le [README](README.md) pour le demarrage.

In [1]:
# Cellule de preparation — executee une fois au demarrage du kernel.
import json
import os
import sys

# Le kernel demarre dans ce dossier (seed_uri) : il y trouve z3_sudoku_core.py.
sys.path.insert(0, os.getcwd())

from z3_sudoku_core import solve_request

# REQUEST n'existe que sous Kernel Gateway (mode notebook-http) ; l'objet est
# injecte au moment de chaque requete. Hors gateway : None -> mode local.
REQUEST = globals().get("REQUEST")

# Puzzle facile du carnet phare Z3-02-Sudoku-Python.ipynb (cellule 3).
DEMO_PUZZLE = (
    "530070000"
    "600195000"
    "098000060"
    "800060003"
    "400803001"
    "700020006"
    "060000280"
    "000419005"
    "000080079"
)

print("Mode :", "Kernel Gateway (notebook-http)" if REQUEST is not None else "local (demonstration)")

Mode : local (demonstration)


## Endpoint `POST /sudoku/solve`

La cellule ci-dessous est annotee en premiere ligne par `# POST /sudoku/solve` :
Kernel Gateway l'execute a chaque requete HTTP, avec l'objet `REQUEST` injecte
dans l'espace de noms. Le JSON imprime devient le corps de la reponse.

In [2]:
# POST /sudoku/solve
# Sous Kernel Gateway, REQUEST porte la requete : chaine JSON en v3
# (double json.dumps cote serveur), dict direct selon les versions -- on
# accepte les deux. Le corps JSON (Content-Type application/json) est
# deja parse dans la cle "body".
requete = None
if isinstance(REQUEST, str):
    try:
        requete = json.loads(REQUEST)
    except Exception:
        requete = None
elif isinstance(REQUEST, dict):
    requete = REQUEST
corps = requete.get("body") if isinstance(requete, dict) else None
puzzle = corps.get("puzzle") if isinstance(corps, dict) else None
if REQUEST is None:
    # Execution locale : demonstration sur le puzzle du carnet phare.
    puzzle = DEMO_PUZZLE
reponse = solve_request(puzzle)
print(json.dumps(reponse))

{"status": "sat", "solution": "534678912672195348198342567859761423426853791713924856961537284287419635345286179", "time_ms": 37.638}


## Execution locale

Sous Kernel Gateway, cette meme cellule repond aux POST ; en execution locale
elle vient de resoudre le puzzle de demonstration. La variante `unsat` (puzzle
contradictoire) est couverte par les tests unitaires de `test_z3_sudoku_core.py`.